In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv("covid_toy.csv")

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [24]:
df["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X = df.drop(["has_covid"], axis=1)
y = df["has_covid"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
66,51,Male,104.0,Mild,Kolkata
55,81,Female,101.0,Mild,Mumbai
58,23,Male,98.0,Strong,Mumbai
44,20,Male,102.0,Strong,Delhi
10,75,Female,NaN,Mild,Delhi
...,...,...,...,...,...
28,16,Male,104.0,Mild,Kolkata
67,65,Male,99.0,Mild,Bangalore
34,74,Male,102.0,Mild,Mumbai
75,5,Male,102.0,Mild,Kolkata


## Without Column Transformer

In [13]:
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[["fever"]])

X_test_fever = si.fit_transform(X_test[["fever"]])

X_train_fever.shape

(80, 1)

In [14]:
# Ordinal Encoding on cough column
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])

X_train_cough = oe.fit_transform(X_train[["cough"]])

X_test_cough = oe.fit_transform(X_test[["cough"]])

X_train_cough.shape

(80, 1)

In [19]:
# OneHotEncoding - gender, city
ohe = OneHotEncoder(drop="first", sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])

X_test_gender_city = ohe.fit_transform(X_test[["gender", "city"]])

X_train_gender_city.shape

(80, 4)

In [25]:
# Extracting Age
X_train_age = X_train.drop(columns=["gender", "fever", "cough", "city"]).values

X_test_age = X_test.drop(columns=["gender", "fever", "cough", "city"]).values

X_train_age.shape

(80, 1)

In [27]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)

X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

X_train_transformed.shape

(80, 7)

## Using Column Transform

In [28]:
from sklearn.compose import ColumnTransformer

In [31]:
transformer = ColumnTransformer(transformers=[
    ("tnf1", SimpleImputer(), ["fever"]),
    ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
    ("tnf3", OneHotEncoder(sparse_output=False, drop="first"), ["gender", "city"])
], remainder="passthrough")

In [32]:
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.fit_transform(X_test).shape

(20, 7)